In [654]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

import os
from enum import Enum

In [655]:
load_dotenv()

True

In [656]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]

MODEL = "google/gemma-4-e2b"

In [657]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

In [658]:
# models = [m.id for m in client.models.list().data]
# models

In [659]:
# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "Hi!"}]
# )

# response.choices[0].message.content

# Modélisation du monde

In [660]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [661]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])
initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# Couche de contrat

In [662]:
class Direction(str, Enum):
    HAUT       = "HAUT"
    BAS        = "BAS"
    GAUCHE     = "GAUCHE"
    DROITE     = "DROITE"


# class Confidence(str, Enum):
#     HIGHT       = "HIGHT"
#     MEDIUM      = "MEDIUM"
#     LOW         = "LOW"


class PlayerDecision(BaseModel):
    direction: Direction
    # playerDecision: str
    # confidence: Confidence

MOVES = {
    "HAUT":     (-1, 0),
    "BAS":      ( 1,  0),
    "GAUCHE":   ( 0,  -1),
    "DROITE":   ( 0,   1),
}

# Moteur de perception

In [663]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [664]:
# localize(initial_map, ENNEMY)

In [665]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
 
    return np.round(distances, 2)

In [666]:
golds_distances = compute_distances(localize(initial_map, GOLD), (1, 1))
ennemies_distances = compute_distances(localize(initial_map, ENNEMY), (1, 1))
print(golds_distances, ennemies_distances)

[5.  6.4] [3.]


In [667]:
def perception(world_map):
    player_position = localize(world_map, PLAYER)[0]
    golds_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    golds_distances = compute_distances(golds_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    nearest_gold_delta = {"row": 0, "col": 0}
    if len(golds_positions) > 0:
        nearest_idx = np.argmin(golds_distances)
        nearest_gold_pos = golds_positions[nearest_idx]
        delta = nearest_gold_pos - player_position
        nearest_gold_delta = {"row": int(delta[0]), "col": int(delta[1])}

    return {
        "ennemies_distances": ennemies_distances.tolist(),
        "ennemies_count": len(ennemies_distances),
        "golds_distances": golds_distances.tolist(),
        "golds_count": len(golds_distances),
        "nearest_gold_delta": nearest_gold_delta,
    }

p = perception(initial_map)
p

{'ennemies_distances': [3.0],
 'ennemies_count': 1,
 'golds_distances': [5.0, 6.4],
 'golds_count': 2,
 'nearest_gold_delta': {'row': 0, 'col': 5}}

In [668]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

In [669]:
# show_map(initial_map)

# Moteur de déplacement

In [670]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] in (VOID, GOLD)

In [671]:
def move(world_map: np.ndarray, old_pos, new_pos):
    if not allowed_move(world_map, new_pos):
        return old_pos
    
    entity = world_map[old_pos[0], old_pos[1]]

    if world_map[old_pos[0], old_pos[1]] == GOLD:
        print(" 💰 Or ramassé !")
        
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    return new_pos

In [672]:
# player_pos = localize(initial_map, PLAYER)[0]
# move(initial_map, player_pos, player_pos + 1)

# Moteur de décision

In [673]:
def decide(player_perception) -> PlayerDecision | None:
    prompt = f"""
    # Contexte
    - Tu es un joueur qui veut maximiser ses gains en or

    # Objectif
    - Trouve le plus court chemin vers l'or

    # Perception
    {player_perception}
    """

    # print(prompt)

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision,
        temperature=1
    )

    return response.choices[0].message.parsed or None

In [674]:
# p = perception(initial_map)
# decision: PlayerDecision = decide(p)
# decision.direction.value

# Game loop (simulation)

In [675]:
def game_loop(world_map: np.ndarray, max_turns=10):
    world_map = world_map.copy()
    move_history = []

    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")
        show_map(world_map)

        player_pos = tuple(localize(world_map, PLAYER)[0])
        p = perception(world_map)
        p["move_history"] = move_history[-5:]  # garder un historique court

        decision = decide(p)
        if decision is not None:
            print(f"\t → LLM decision: {decision.direction.value}")

            d_row, d_col = MOVES[decision.direction.value]
            new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
            result_pos = move(world_map, player_pos, new_pos)

            moved = result_pos != player_pos
            move_history.append(f"{decision.direction.value} ({'ok' if moved else 'BLOQUÉ - mur ou hors limites'})")

In [676]:
game_loop(world_map=initial_map, max_turns=30)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 → LLM decision: DROITE

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 → LLM decision: DROITE

 =================== [Turn 3] ===================
·	·	·	·	·	·	·
·	·	·	👤	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 → LLM decision: DROITE

 =================== [Turn 4] ===================
·	·	·	·	·	·	·
·	·	·	👤	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
-----------------------------------------------------
	 → LLM decision: HAUT

 =================== [Turn 5] ===================
·	·	·	👤	·	·	·
·	·	·	·	👹	·	💰
·	·	·	·	·	·